In [12]:
import os
from pathlib import Path
from pprint import pprint

from datasets import load_dataset
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
from openai import OpenAI

from speeling_good.cli import prepare_fineweb
from speeling_good.preprocessing.llm_annotation_provider import (
    AnnotationConfig,
    LlmAnnotationProvider,
)
from speeling_good.preprocessing.types import RawPassage

In [13]:
fw = load_dataset(
    "HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=True
)
fw = fw.select_columns(["text"])

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

In [14]:
client = OpenAI(
    base_url=os.environ["MODEL_ENDPOINT"] + "/v1/", api_key=os.getenv("MODEL_TOKEN")
)

config_dir = Path(prepare_fineweb.__file__).resolve().parents[3] / "config"
with initialize_config_dir(version_base="1.3", config_dir=str(config_dir)):
    config = compose(config_name="prepare_fineweb")
annotation = AnnotationConfig.model_validate(
    OmegaConf.to_container(config.annotation, resolve=True)
)
annotator = LlmAnnotationProvider(client=client, config=annotation)

In [15]:
for row in fw.take(10):
    passage = RawPassage(text=row["text"])
    annotated = annotator.annotate(passage)
    pprint(annotated)
    print("\n" * 2)

AnnotatedPassage(text='|Viewing Single Post From: Spoilers for the Week of '
                      'February 11th|\n'
                      '|Lil||Feb 1 2013, 09:58 AM|\n'
                      "Don't care about Chloe/Taniel/Jen-Jen. Don't care about "
                      'Sami, really, but hoping that we get some good '
                      '"SAMANTHA GENE!!" Marlena Death-Stares out of it. And '
                      '"newfound" feelings. Please. If only.\n'
                      'STEFANO!! STEFANO, STEFANO, STEFANO!!!! :cheer:\n'
                      '|Spoilers for the Week of February 11th · DAYS: News, '
                      'Spoilers & Discussion|',
                 annotations=[])





2026-11-11 13:04:50.118 | WARNING  | speeling_good.preprocessing.llm_annotation_provider:_retry_or_raise:247 - Annotation attempt 1/3 failed (Each correction's exact context must match once in the passage; copy adjacent context verbatim and extend ambiguous context); retrying in 0.81s
2026-11-11 13:05:49.843 | WARNING  | speeling_good.preprocessing.llm_annotation_provider:_retry_or_raise:247 - Annotation attempt 2/3 failed (The response must be complete); retrying in 1.49s
2026-11-11 13:06:50.320 | ERROR    | speeling_good.preprocessing.llm_annotation_provider:_retry_or_raise:239 - Annotation failed after 3 attempts (The response must be complete)


AnnotationError: No valid annotation after 3 attempts